[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mperumal-usd/capstone_team_3/blob/main/notebooks/COLAB_CNN_MEL_Similarity.ipynb)

# CNN Mel-Spectrogram Similarity Search

Embeds audio chunks using **mel spectrograms → pretrained ResNet18 CNN** (no MERT).
Indexes all classic ChunkSamples, then queries with TestSamples chunks via FAISS.

## Sections
- **Section 0** — Setup (install deps, mount Drive, config)
- **Section 1** — Mel spectrogram + CNN embedding pipeline
- **Section 2** — Build classic chunk embeddings (index database)
- **Section 3** — FAISS index construction
- **Section 4** — Build test chunk embeddings
- **Section 5** — Similarity search & results
- **Section 6** — Recall@K evaluation (requires TestingReferences.csv)

In [ ]:
# ── Section 0: Setup ──────────────────────────────────────────────────────────
!pip install -q librosa soundfile faiss-cpu tqdm numpy pandas scikit-learn matplotlib seaborn

from google.colab import drive
drive.mount('/content/drive')

import os, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms as transforms
import librosa
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# ── Config ────────────────────────────────────────────────────────────────────
DRIVE_BASE      = "/content/drive/MyDrive/AAI-590 Capstone/checkpoints_cnn_mel"
CHUNK_PATH      = "/content/drive/MyDrive/AAI-590 Capstone/AAI590_FinalProject/ChunkSamples"
TEST_CHUNK_DIR  = "/content/drive/MyDrive/AAI-590 Capstone/AAI590_FinalProject/MidiDatasets/TestingSamples/TestChunks"
REF_CSV_PATH    = "/content/TestingReferences.csv"   # upload to Colab if available

# Mel spectrogram parameters
SR          = 24000
N_MELS      = 128
N_FFT       = 1024
HOP_LENGTH  = 512
MEL_H       = 128   # height (mel bins) after resize
MEL_W       = 128   # width  (time frames) after resize

# CNN / embedding parameters
EMBED_DIM   = 512   # ResNet18 avg-pool output dimension
BATCH_SIZE  = 32
SEED        = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Create output directories
for sub in ["results"]:
    os.makedirs(os.path.join(DRIVE_BASE, sub), exist_ok=True)
print("Drive folders ready.")

## Section 1 — Mel Spectrogram + CNN Embedding Pipeline

### Architecture
```
WAV (24 kHz)
  → librosa.feature.melspectrogram  (128 mel bins)
  → power_to_db + normalize → [0, 1]
  → resize to 128×128
  → repeat channel ×3  (grayscale → RGB-like)
  → ResNet18 (pretrained, FC removed)
  → 512-dim global avg-pool feature
  → L2-normalize → 512-dim unit-sphere embedding
```

In [ ]:
# ── Section 1A: Mel spectrogram helper ────────────────────────────────────────

def wav_to_melspec_tensor(wav_path: str) -> torch.Tensor:
    """
    Load a WAV file and return a (3, MEL_H, MEL_W) float32 tensor
    ready for the CNN.

    Steps:
      1. Load audio at target SR.
      2. Compute mel spectrogram (power).
      3. Convert to dB and min-max normalize to [0, 1].
      4. Resize to (MEL_H, MEL_W) via PIL bicubic.
      5. Repeat grayscale channel 3× for RGB-style input.
    """
    audio, _ = librosa.load(wav_path, sr=SR, mono=True)
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)   # shape: (n_mels, time)

    # Normalize to [0, 1]
    mel_min, mel_max = mel_db.min(), mel_db.max()
    if mel_max - mel_min > 1e-6:
        mel_norm = (mel_db - mel_min) / (mel_max - mel_min)
    else:
        mel_norm = np.zeros_like(mel_db)

    # Resize to (MEL_H, MEL_W) using PIL
    img = Image.fromarray((mel_norm * 255).astype(np.uint8), mode='L')
    img = img.resize((MEL_W, MEL_H), Image.BICUBIC)   # PIL: (width, height)
    mel_resized = np.array(img).astype(np.float32) / 255.0  # (MEL_H, MEL_W)

    # (1, H, W) → (3, H, W) by repeating the channel
    tensor = torch.from_numpy(mel_resized).unsqueeze(0).repeat(3, 1, 1)
    return tensor


# ── Section 1B: CNN model (ResNet18 without head) ─────────────────────────────

class MelCNNEmbedder(nn.Module):
    """
    Pretrained ResNet18 with the final FC layer replaced by identity.
    Outputs a 512-dim L2-normalized embedding per mel spectrogram.
    """
    def __init__(self):
        super().__init__()
        backbone = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        # Drop the classification head; keep up to avgpool → 512-dim
        self.features = nn.Sequential(*list(backbone.children())[:-1])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, 3, H, W)
        feat = self.features(x)          # (batch, 512, 1, 1)
        feat = feat.flatten(1)           # (batch, 512)
        return F.normalize(feat, p=2, dim=1)  # L2-normalized


# ImageNet normalization for ResNet
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def normalize_imagenet(t: torch.Tensor) -> torch.Tensor:
    """Apply ImageNet mean/std normalization to a (3, H, W) tensor."""
    return (t - IMAGENET_MEAN) / IMAGENET_STD


cnn_model = MelCNNEmbedder().to(DEVICE)
cnn_model.eval()
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f"MelCNNEmbedder loaded. Total params: {total_params:,}  Embed dim: {EMBED_DIM}")


# ── Section 1C: Sanity-check on a single file ──────────────────────────────────
def quick_sanity_check():
    """Find the first .wav in ChunkSamples and check the full pipeline."""
    for root, _, files in os.walk(CHUNK_PATH):
        for f in files:
            if f.endswith('.wav'):
                path = os.path.join(root, f)
                print(f"Sanity check on: {path}")
                t = wav_to_melspec_tensor(path)           # (3, H, W)
                print(f"  Mel tensor shape : {tuple(t.shape)}  range [{t.min():.3f}, {t.max():.3f}]")
                with torch.no_grad():
                    emb = cnn_model(normalize_imagenet(t).unsqueeze(0).to(DEVICE))
                print(f"  Embedding shape  : {tuple(emb.shape)}  L2-norm: {emb.norm().item():.4f}")

                # Visualize the mel spectrogram
                mel_np = t[0].numpy()  # (H, W) grayscale
                plt.figure(figsize=(8, 4))
                plt.imshow(mel_np, aspect='auto', origin='lower', cmap='magma')
                plt.colorbar(label='Normalized dB')
                plt.title(f'Mel Spectrogram — {f}')
                plt.xlabel('Time frames')
                plt.ylabel('Mel bins')
                plt.tight_layout()
                plt.show()
                return

quick_sanity_check()

## Section 2 — Build Classic Chunk Embeddings

Walks `ChunkSamples/` (structure: `composer/song/chunk_N.wav`),
computes mel spectrogram → CNN embedding for every chunk, and saves to pickle.

**Skip cell if `classic_embeddings.pkl` already exists on Drive.**

In [ ]:
# ── Section 2: Classic chunk embeddings ───────────────────────────────────────

CLASSIC_EMB_PATH = os.path.join(DRIVE_BASE, "results", "classic_embeddings.pkl")

if os.path.exists(CLASSIC_EMB_PATH):
    print(f"Found existing embeddings → loading from {CLASSIC_EMB_PATH}")
    classic_df = pd.read_pickle(CLASSIC_EMB_PATH)
    print(f"Loaded {len(classic_df)} classic chunk embeddings.")
    print(classic_df.head())
else:
    # Collect all WAV paths
    classic_records = []   # [{filename, composer, song, chunk, path}]
    for composer in sorted(os.listdir(CHUNK_PATH)):
        composer_path = os.path.join(CHUNK_PATH, composer)
        if not os.path.isdir(composer_path):
            continue
        for song in sorted(os.listdir(composer_path)):
            song_path = os.path.join(composer_path, song)
            if not os.path.isdir(song_path):
                continue
            for f in sorted(os.listdir(song_path)):
                if not f.endswith('.wav'):
                    continue
                m = re.search(r'_chunk_(\d+)\.wav$', f)
                chunk_num = int(m.group(1)) if m else -1
                classic_records.append({
                    'filename': f,
                    'composer': composer,
                    'song': song,
                    'chunk': chunk_num,
                    'path': os.path.join(song_path, f),
                })

    print(f"Found {len(classic_records)} classic chunks across {len(set(r['composer'] for r in classic_records))} composers.")

    # ── Dataset & DataLoader ──────────────────────────────────────────────────
    class MelDataset(Dataset):
        def __init__(self, records):
            self.records = records

        def __len__(self):
            return len(self.records)

        def __getitem__(self, idx):
            rec = self.records[idx]
            try:
                tensor = wav_to_melspec_tensor(rec['path'])   # (3, H, W)
                tensor = normalize_imagenet(tensor)
                ok = True
            except Exception:
                tensor = torch.zeros(3, MEL_H, MEL_W)
                ok = False
            return tensor, idx, ok

    def collate_fn(batch):
        tensors, idxs, oks = zip(*batch)
        return torch.stack(tensors), list(idxs), list(oks)

    dataset = MelDataset(classic_records)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=2, pin_memory=True)

    # ── Extract embeddings ────────────────────────────────────────────────────
    embeddings_map = {}   # idx → embedding np array
    cnn_model.eval()

    with torch.no_grad():
        for tensors, idxs, oks in tqdm(loader, desc="Embedding classic chunks"):
            tensors = tensors.to(DEVICE)
            embs = cnn_model(tensors).cpu().numpy()  # (batch, 512)
            for i, (idx, ok) in enumerate(zip(idxs, oks)):
                if ok:
                    embeddings_map[idx] = embs[i]

    # Build DataFrame
    rows = []
    for idx, rec in enumerate(classic_records):
        if idx in embeddings_map:
            rows.append({**rec, 'embedding': embeddings_map[idx]})

    classic_df = pd.DataFrame(rows)
    classic_df.to_pickle(CLASSIC_EMB_PATH)
    print(f"\nSaved {len(classic_df)} classic embeddings → {CLASSIC_EMB_PATH}")
    print(classic_df[['filename', 'composer', 'song', 'chunk']].head())

## Section 3 — FAISS Index Construction

Builds a **FAISS `IndexFlatIP`** (inner-product / cosine on L2-normalized vectors)
from all classic chunk embeddings.

In [ ]:
# ── Section 3: FAISS index ─────────────────────────────────────────────────────
!pip install -q faiss-cpu
import faiss

# Stack embeddings (float32 contiguous required by FAISS)
classic_embs = np.vstack(classic_df['embedding'].values).astype('float32')
print(f"classic_embs shape: {classic_embs.shape}")

# IndexFlatIP = cosine similarity search (vectors already L2-normalized)
d = EMBED_DIM
faiss_index = faiss.IndexFlatIP(d)
faiss_index.add(classic_embs)
print(f"FAISS IndexFlatIP built. Vectors indexed: {faiss_index.ntotal}")

# Save index to Drive for reuse
INDEX_PATH = os.path.join(DRIVE_BASE, "results", "classic_faiss.index")
faiss.write_index(faiss_index, INDEX_PATH)
print(f"Index saved → {INDEX_PATH}")

## Section 4 — Build Test Chunk Embeddings

Processes all `.wav` files under `TestingSamples/TestChunks/` using the same
mel spectrogram → CNN pipeline.

In [ ]:
# ── Section 4: Test chunk embeddings ──────────────────────────────────────────

TEST_EMB_PATH = os.path.join(DRIVE_BASE, "results", "test_embeddings.pkl")

if os.path.exists(TEST_EMB_PATH):
    print(f"Found existing test embeddings → loading from {TEST_EMB_PATH}")
    test_df = pd.read_pickle(TEST_EMB_PATH)
    print(f"Loaded {len(test_df)} test chunk embeddings.")
    print(test_df.head())
else:
    # Collect all test WAV paths
    test_records = []
    for root, dirs, files in os.walk(TEST_CHUNK_DIR):
        for f in sorted(files):
            if not f.endswith('.wav'):
                continue
            m = re.search(r'_chunk_(\d+)\.wav$', f)
            chunk_num = int(m.group(1)) if m else -1
            test_records.append({
                'filename': f,
                'song': os.path.basename(root),
                'chunk': chunk_num,
                'path': os.path.join(root, f),
            })

    print(f"Found {len(test_records)} test chunks in {TEST_CHUNK_DIR}")

    class MelDataset(Dataset):
        def __init__(self, records):
            self.records = records

        def __len__(self):
            return len(self.records)

        def __getitem__(self, idx):
            rec = self.records[idx]
            try:
                tensor = wav_to_melspec_tensor(rec['path'])
                tensor = normalize_imagenet(tensor)
                ok = True
            except Exception:
                tensor = torch.zeros(3, MEL_H, MEL_W)
                ok = False
            return tensor, idx, ok

    def collate_fn(batch):
        tensors, idxs, oks = zip(*batch)
        return torch.stack(tensors), list(idxs), list(oks)

    test_dataset = MelDataset(test_records)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)

    test_emb_map = {}
    cnn_model.eval()
    with torch.no_grad():
        for tensors, idxs, oks in tqdm(test_loader, desc="Embedding test chunks"):
            tensors = tensors.to(DEVICE)
            embs = cnn_model(tensors).cpu().numpy()
            for i, (idx, ok) in enumerate(zip(idxs, oks)):
                if ok:
                    test_emb_map[idx] = embs[i]

    rows = []
    for idx, rec in enumerate(test_records):
        if idx in test_emb_map:
            rows.append({**rec, 'embedding': test_emb_map[idx]})

    test_df = pd.DataFrame(rows)
    test_df.to_pickle(TEST_EMB_PATH)
    print(f"\nSaved {len(test_df)} test embeddings → {TEST_EMB_PATH}")
    print(test_df[['filename', 'song', 'chunk']].head())

## Section 5 — Similarity Search

Queries the FAISS index with every test chunk and retrieves the **top-10 most similar**
classic chunks. Reports inner-product similarity (= cosine similarity for L2-normalized vectors).

In [ ]:
# ── Section 5A: Run FAISS search ──────────────────────────────────────────────

K = 10  # top-K matches

test_embs = np.vstack(test_df['embedding'].values).astype('float32')
print(f"Querying FAISS with {len(test_embs)} test chunks (top-{K} matches)...")

scores, indices = faiss_index.search(test_embs, K)  # scores = cosine similarity

# Build results DataFrame
results_rows = []
for i, row in test_df.reset_index(drop=True).iterrows():
    entry = {
        'Query_File': row['filename'],
        'Query_Song': row['song'],
        'Query_Chunk': row['chunk'],
    }
    for rank in range(K):
        match_idx  = int(indices[i][rank])
        match_row  = classic_df.iloc[match_idx]
        cos_sim    = float(scores[i][rank])
        entry[f'Rank_{rank+1}_Match']    = match_row['filename']
        entry[f'Rank_{rank+1}_Song']     = match_row['song']
        entry[f'Rank_{rank+1}_Composer'] = match_row['composer']
        entry[f'Rank_{rank+1}_Sim']      = round(cos_sim, 4)
    results_rows.append(entry)

results_df = pd.DataFrame(results_rows)

RESULTS_CSV = os.path.join(DRIVE_BASE, "results", "test_search_results.csv")
results_df.to_csv(RESULTS_CSV, index=False)
print(f"Saved search results → {RESULTS_CSV}")
print(f"\nSample results (first 5 queries, top-3 matches):")

display_cols = ['Query_File', 'Query_Song',
                'Rank_1_Song', 'Rank_1_Composer', 'Rank_1_Sim',
                'Rank_2_Song', 'Rank_2_Composer', 'Rank_2_Sim',
                'Rank_3_Song', 'Rank_3_Composer', 'Rank_3_Sim']
display(results_df[display_cols].head())

In [ ]:
# ── Section 5B: Visualize similarity distribution ────────────────────────────

rank1_sims = results_df['Rank_1_Sim'].values
rank5_sims = results_df['Rank_5_Sim'].values

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(rank1_sims, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(rank1_sims.mean(), color='tomato', linestyle='--', linewidth=2,
                label=f'Mean = {rank1_sims.mean():.3f}')
axes[0].set_title('Rank-1 Cosine Similarity (Test → Classic)')
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(rank5_sims, bins=50, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[1].axvline(rank5_sims.mean(), color='tomato', linestyle='--', linewidth=2,
                label=f'Mean = {rank5_sims.mean():.3f}')
axes[1].set_title('Rank-5 Cosine Similarity (Test → Classic)')
axes[1].set_xlabel('Cosine Similarity')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_BASE, "results", "similarity_distribution.png"), dpi=150)
plt.show()

# Per-song mean similarity (top-1)
per_song = results_df.groupby('Query_Song')['Rank_1_Sim'].mean().sort_values(ascending=False)
print("\nMean Rank-1 cosine similarity per test song (top 15):")
print(per_song.head(15).to_string())

In [ ]:
# ── Section 5C: Side-by-side mel spec visualization (query vs top-1 match) ───

import random
random.seed(SEED)

sample_indices = random.sample(range(len(test_df)), min(4, len(test_df)))

fig, axes = plt.subplots(len(sample_indices), 2,
                         figsize=(14, 4 * len(sample_indices)))

for row_i, qi in enumerate(sample_indices):
    query_row  = test_df.iloc[qi]
    result_row = results_df.iloc[qi]

    match_idx   = int(indices[qi][0])
    match_rec   = classic_df.iloc[match_idx]

    # Load mel specs
    q_mel  = wav_to_melspec_tensor(query_row['path'])[0].numpy()
    m_mel  = wav_to_melspec_tensor(match_rec['path'])[0].numpy()

    ax_q = axes[row_i][0]
    ax_m = axes[row_i][1]

    ax_q.imshow(q_mel, aspect='auto', origin='lower', cmap='magma')
    ax_q.set_title(f"Query: {query_row['filename']}\n(song: {query_row['song']})", fontsize=9)
    ax_q.set_xlabel('Time frames'); ax_q.set_ylabel('Mel bins')

    ax_m.imshow(m_mel, aspect='auto', origin='lower', cmap='magma')
    cos_sim = result_row['Rank_1_Sim']
    ax_m.set_title(
        f"Top-1 Match: {match_rec['filename']}\n"
        f"(composer: {match_rec['composer']} | sim: {cos_sim:.4f})",
        fontsize=9
    )
    ax_m.set_xlabel('Time frames')

plt.suptitle('Query Mel Spectrograms vs Top-1 Classic Matches', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_BASE, "results", "mel_spec_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()

## Section 6 — Recall@K Evaluation

Requires `TestingReferences.csv` uploaded to `/content/TestingReferences.csv`.
Columns expected: `FileName` (test file), `OriginalFileName` (ground-truth classic song stem).

Skip this section if the reference CSV is not available.

In [ ]:
# ── Section 6: Recall@K evaluation ───────────────────────────────────────────

if not os.path.exists(REF_CSV_PATH):
    print(f"Reference CSV not found at {REF_CSV_PATH} — skipping Recall@K evaluation.")
else:
    ref_df = pd.read_csv(REF_CSV_PATH)
    print(f"Loaded reference CSV: {len(ref_df)} rows")
    display(ref_df.head())

    col_query = 'FileName'
    col_truth = 'OriginalFileName'

    # Build mapping: cleaned test song base → truth classic song stem
    truth_mapping = {}
    for q, t in zip(ref_df[col_query], ref_df[col_truth]):
        if pd.isna(q) or pd.isna(t):
            continue
        clean_q = str(q).rsplit('.', 1)[0]  # strip extension
        clean_t = str(t).rsplit('.', 1)[0]
        truth_mapping[clean_q] = clean_t

    recall_at_k = {1: 0, 3: 0, 5: 0, 10: 0}
    total_evaluated = 0

    for _, row in results_df.iterrows():
        query_file = row['Query_File']
        base_query = query_file.rsplit('_chunk_', 1)[0]

        if base_query not in truth_mapping:
            continue

        true_song = truth_mapping[base_query]
        total_evaluated += 1

        predicted_songs = [
            row[f'Rank_{rank}_Song'] for rank in range(1, 11)
        ]

        for k in recall_at_k:
            if true_song in predicted_songs[:k]:
                recall_at_k[k] += 1

    print(f"\nTotal valid queries evaluated: {total_evaluated}")
    recall_data = []
    for k, hits in recall_at_k.items():
        recall = hits / total_evaluated if total_evaluated > 0 else 0.0
        recall_data.append({'K': k, 'Recall': recall})
        print(f"Recall@{k:02d}: {recall:.4f}")

    recall_df = pd.DataFrame(recall_data)

    RECALL_CSV = os.path.join(DRIVE_BASE, "results", "recall_at_k.csv")
    recall_df.to_csv(RECALL_CSV, index=False)
    print(f"Recall CSV saved → {RECALL_CSV}")

    # Plot
    plt.figure(figsize=(7, 5))
    sns.barplot(x='K', y='Recall', data=recall_df, palette='viridis')
    plt.title('CNN Mel-Spectrogram: Recall@K (Test Set)')
    plt.xlabel('K')
    plt.ylabel('Recall')
    plt.ylim(0, 1.05)
    for idx, r in recall_df.iterrows():
        plt.text(idx, r.Recall + 0.02, f"{r.Recall:.4f}", ha='center', fontweight='bold')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_BASE, "results", "recall_at_k.png"), dpi=150)
    plt.show()